In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [2]:
random.seed(28)
np.random.seed(28)

### User

In [3]:
num_users = 20000
start_date = datetime(2025, 1, 1)
channels = ['TikTok_Ads', 'Baidu_SEM', 'TapTap', 'AppStore', 'Friend_Referral']
weights = [0.35, 0.15, 0.25, 0.15, 0.10]
data = {
    'user_id': range(1000001, 1000001 + num_users),
    'install_date': [start_date + timedelta(days=random.randint(0, 30)) for _ in range(num_users)],
    'channel': np.random.choice(channels, num_users, p=weights)
}
df_user = pd.DataFrame(data)

In [4]:
# 传播
df_user = df_user.sort_values('install_date').reset_index(drop=True)
inviter_ids = []
existing_user_pool = []
for index, row in df_user.iterrows():
    if row['channel'] == 'Friend_Referral' and len(existing_user_pool) > 0:
        inviter_id = random.choice(existing_user_pool)
        inviter_ids.append(inviter_id)
    else:
        inviter_ids.append(None)
        if row['channel'] == 'Friend_Referral': 
            df_user.at[index, 'channel'] = random.choice(['TikTok_Ads', 'AppStore'])
    existing_user_pool.append(row['user_id'])

df_user['inviter_id'] = inviter_ids
df_user['inviter_id'] = df_user['inviter_id'].astype('Int64')

In [5]:
# 激活
# 激活：“完成新手教程”
activation_probs = {
    'TikTok_Ads': 0.55,
    'Baidu_SEM': 0.4,   
    'TapTap': 0.75, 
    'AppStore': 0.65,
    'Friend_Referral': 0.85
}
df_user['activated'] = df_user['channel'].apply(lambda x: 1 if random.random() < activation_probs[x] else 0)

### Orders 充值

In [6]:
orders = []
order_id_counter = 202501010000

# 付费率设置 (Pay Rate)
pay_probs= {
    'TikTok_Ads': 0.04,
    'Baidu_SEM': 0.03,
    'TapTap': 0.08,
    'AppStore': 0.07,
    'Friend_Referral': 0.15 
}

price_tiers = [6, 30, 68, 128, 328, 648]
# 普通玩家：月卡/大月卡
normal_tier_weights = [0.35, 0.45, 0.15, 0.03, 0.01, 0.01]
# 大R玩家：更倾向于328，648
whale_tier_weights = [0.01, 0.01, 0.05, 0.15, 0.30, 0.48]
for index, row in df_user.iterrows():
    if row['activated'] == 1:
        # 判断是否转化为付费用户
        if random.random() < pay_probs[row['channel']]:
            user_id = row['user_id']
            install_date = row['install_date']
            
            # 鲸鱼判断 (5% 的付费用户是鲸鱼)
            is_whale = random.random() < 0.02
            
            if is_whale:
                num_orders = int(np.random.exponential(scale=8)) + 5
                if num_orders > 50:
                    num_orders = 50
                tier_weights = whale_tier_weights
            else:
                num_orders = np.random.choice([1, 2, 3, 4], p=[0.6, 0.25, 0.1, 0.05])
                tier_weights = normal_tier_weights
            for _ in range(num_orders):
                amount = np.random.choice(price_tiers, p=tier_weights)
                
                days_after_install = int(np.random.exponential(scale=10))
                if days_after_install > 60: days_after_install = 60
                order_time = install_date + timedelta(days=days_after_install, hours=random.randint(0,23))
                
                orders.append([order_id_counter, user_id, order_time, amount])
                order_id_counter += 1

df_orders = pd.DataFrame(orders, columns=['order_id', 'user_id', 'order_time', 'amount'])
print(df_orders['amount'].value_counts(normalize=True).sort_index())

total_revenue = df_orders['amount'].sum()
revenue_by_tier = df_orders.groupby('amount')['amount'].sum() / total_revenue
print(revenue_by_tier)

amount
6      0.277716
30     0.371397
68     0.120843
128    0.053215
328    0.073171
648    0.103659
Name: proportion, dtype: float64
amount
6      0.014002
30     0.093623
68     0.069048
128    0.057236
328    0.201668
648    0.564423
Name: amount, dtype: float64


### Login Log

In [7]:
login_records = set()

# 充值日必须登录
for index, row in df_orders.iterrows():
    login_records.add((row['user_id'], row['order_time'].date()))

# 活跃留存逻辑
for index, row in df_user.iterrows():
    user_id = row['user_id']
    install_date = row['install_date'].date()
    
    # 注册当天必登录
    login_records.add((user_id, install_date))
    
    if row['activated'] == 1:
        base_retention = 0.6
        if row['channel'] == 'Friend_Referral': base_retention = 0.80 # 邀请用户留存高
        
        for day_offset in range(1, 61):
            prob = base_retention / (day_offset ** 0.8) 
            
            if random.random() < prob:
                login_date = install_date + timedelta(days=day_offset)
                login_records.add((user_id, login_date))
    else:
        # 未激活用户，只有极低概率第二天再登一下
        if random.random() < 0.02:
            login_date = install_date + timedelta(days=1)
            login_records.add((user_id, login_date))

df_logins = pd.DataFrame(list(login_records), columns=['user_id', 'login_date'])
df_logins = df_logins.sort_values(['user_id', 'login_date']).reset_index(drop=True)

In [8]:
df_user.to_csv('game_user_data.csv', index=False)
df_orders.to_csv('game_iap_data.csv', index=False)
df_logins.to_csv('game_login_data.csv', index=False)